# Finetune MicroSAM

Finetune the microsam `vit_b_lm` encoder and AIS decoder on labeled data.

note: due to limit on VRAM and the 1660Ti card. The finetuning has moved to kaggle.

## Load training data

In [ ]:
# make local editable packages automatically reload
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from image_processing_tools.image_class.micro_sam_pipeline import load_config
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path

config = load_config('micro_sam_config.json')
config['preprocessing']['resize_image'] = False
config['preprocessing']['max_dim'] = 1024
config['preprocessing']['quantization'] = "8bit"
config['preprocessing']['two_channel_merge_mode'] = "passthrough"

with open("microsam_finetune_data.json") as f:
    data = json.load(f)
    
samples = data["samples"]

images = []
labels = []
for sample in samples:
    label_path = Path(sample["label"]).expanduser()
    
    image_channels = sample["image"]          # dict of channel_name → path
    dic_path = Path(image_channels["dic"]).expanduser()
    fluo_path = image_channels.get("fluorescence")  # None if not present
    fluo_path = Path(fluo_path).expanduser() if fluo_path is not None else fluo_path
    dapi_path = image_channels.get("dapi")        # None if not present
    dapi_path = Path(dapi_path).expanduser() if dapi_path is not None else dapi_path
    
    dic_shift = sample["correct_DIC_shift"]
    config['preprocessing']['correct_DIC_shift'] = dic_shift
    
    labels.append(ImageContainer(label_path, config, is_label=True))
    image = ImageContainer([fluo_path,dapi_path,dic_path], config) if fluo_path is not None and dapi_path is not None else ImageContainer(dic_path, config)
    images.append(image)

In [ ]:
import matplotlib.pyplot as plt
from torch_em.util.util import get_random_colors

fig,axes = plt.subplots(1,2)

images[1].display(ax=axes[0])
labels[1].display(ax=axes[1],cmap=get_random_colors(labels[4].merge()))

## Create data loaders

In [ ]:
import torch
import torch_em
import numpy as np
import kornia.augmentation as K
from micro_sam.training import default_sam_loader
from micro_sam.training.util import require_8bit
from torch_em.transform.augmentation import KorniaAugmentationPipeline
import tempfile, os, tifffile

def save_to_tempdir(raw_list, label_list, prefix):
    tmpdir = tempfile.mkdtemp(prefix=f"microsam_{prefix}_")
    raw_paths, label_paths = [], []
    for i, (raw, label) in enumerate(zip(raw_list, label_list)):
        rp = os.path.join(tmpdir, f"raw_{i}.tif")
        lp = os.path.join(tmpdir, f"label_{i}.tif")
        tifffile.imwrite(rp, raw)
        tifffile.imwrite(lp, label)
        raw_paths.append(rp)
        label_paths.append(lp)
    return raw_paths, label_paths, tmpdir

def to_chw(arr):
    if arr.ndim == 2:
        return np.stack([arr, arr, arr], axis=0)
    return arr.transpose(2, 0, 1)

class SplitPhotometricPipeline(torch.nn.Module):
    def __init__(self, geometric_augs, photometric_augs):
        super().__init__()
        self.geometric = KorniaAugmentationPipeline(*geometric_augs)
        self.photometric = KorniaAugmentationPipeline(*photometric_augs)

    def forward(self, raw, labels):
        raw, labels = self.geometric(raw, labels)   # flips + scale
        # 90° rotation: same k applied to both
        k = torch.randint(0, 4, (1,)).item()
        raw   = torch.rot90(raw,   k, dims=[-2, -1])
        labels = torch.rot90(labels, k, dims=[-2, -1])
        raw, = self.photometric(raw)
        raw = raw.clamp(0, 255)
        return raw, labels

raw_arrays   = [to_chw(img.merge()) for img in images]
label_arrays = [lbl.merge() for lbl in labels]

n_val = 1
raw_train,   raw_val   = raw_arrays[:-n_val],   raw_arrays[-n_val:]
label_train, label_val = label_arrays[:-n_val], label_arrays[-n_val:]

raw_paths_train, label_paths_train, train_tmp_dir = save_to_tempdir(raw_train, label_train, "train")
raw_paths_val,   label_paths_val, val_tmp_dir   = save_to_tempdir(raw_val,   label_val,   "val")

train_transform = SplitPhotometricPipeline(
    geometric_augs=[
        K.RandomHorizontalFlip(p=0.5),
        K.RandomVerticalFlip(p=0.5),
        K.RandomAffine(degrees=0, scale=(0.85, 1.15), p=0.5),
    ],
    photometric_augs=[
        K.RandomGaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.0), p=0.5),
        # K.ColorJitter(brightness=0.01, contrast=0.1, p=0.5),
    ],
)

shared_kwargs = dict(
    raw_key=None,
    label_key=None,
    patch_shape=(512, 512), # try (256,256)
    with_segmentation_decoder=True,
    train_instance_segmentation_only=True,
    with_channels=True,
    batch_size=1,
    num_workers=2,
    sampler = torch_em.data.sampler.MinInstanceSampler(2,min_size=25)
)

train_loader = default_sam_loader(
    raw_paths=raw_paths_train,
    label_paths=label_paths_train,
    shuffle=True,
    is_train=True,
    transform=train_transform,
    raw_transform=require_8bit,
    **shared_kwargs,
)

val_loader = default_sam_loader(
    raw_paths=raw_paths_val,
    label_paths=label_paths_val,
    shuffle=False,
    is_train=False,
    raw_transform=require_8bit,
    **shared_kwargs,
)

In [ ]:
train_tmp_dir

In [ ]:
from torch_em.util.debug import check_loader

check_loader(train_loader,4,plt=True)

## Finetuning

FP16 is causing overflow in the forward pass, producing NaN losses, but FP32 is fine. However, training with FP32 causes OOM error. So bfloat16 is used.

In [ ]:
import functools
import torch
from functools import partial
import torch_em.trainer.default_trainer as _dt

_orig_init = _dt.DefaultTrainer.__init__

@functools.wraps(_orig_init)
def _patched_init_bf16(self, *args, **kwargs):
    kwargs["mixed_precision"] = True      # keep mixed-precision code paths
    _orig_init(self, *args, **kwargs)
    self.scaler = None                    # bfloat16 needs no gradient scaling
    device_type = "cpu" if self.device.type == "cpu" else "cuda"
    bf16 = partial(torch.autocast, device_type=device_type, dtype=torch.bfloat16)
    self._train_epoch_mixed = lambda progress: self._train_epoch_impl(progress, bf16,
self._backprop)
    self._validate_mixed   = lambda: self._validate_impl(bf16)

_dt.DefaultTrainer.__init__ = _patched_init_bf16

In [ ]:
from micro_sam.util import get_device
from micro_sam.training import train_instance_segmentation
from pathlib import Path

device = get_device()

save_root = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam_finetune").expanduser()

encoder_path = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm").expanduser()
decoder_path = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_b_lm_decoder").expanduser()

# train_instance_segmentation(
#     name="microsam_candida",
#     model_type="vit_b_lm",
#     train_loader=train_loader,
#     val_loader=val_loader,
#     checkpoint_path=encoder_path,
#     n_epochs=100,
#     save_root=save_root,
#     device=device,
#     freeze=["image_encoder"],
#     decoder_path=decoder_path,
# )

In [ ]:
# import torch
# torch.cuda.memory.empty_cache()

## Debug FP16 error

On 1660Ti GPU, using FP16 in training resulted in an error. See [issue](https://github.com/computational-cell-analytics/micro-sam/issues/1214). The code below investigates the reason.

### Find where the NaNs come from

In [ ]:
from micro_sam.training.util import get_trainable_sam_model
from micro_sam.instance_segmentation import get_unetr

device = get_device()
sam_model, state = get_trainable_sam_model(
    model_type="vit_b_lm", device=device,
    checkpoint_path=encoder_path, return_state=True, decoder_path=decoder_path,
)
model = get_unetr(
    image_encoder=sam_model.sam.image_encoder,
    decoder_state=state.get("decoder_state", None),
    device=device,
).eval()

for raw, label in train_loader:
    raw = raw.to(device)

    # 1. full fp32
    with torch.no_grad():
        pred_fp32 = model(raw)
    print("fp32 full  :", pred_fp32.min().item(), pred_fp32.max().item(), "nan:", torch.isnan(pred_fp32).any().item())

    # 2. encoder only in fp16
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        x_pre, _ = model.preprocess(raw)
        encoder_outputs = model.encoder(x_pre)
    for i, feat in enumerate(encoder_outputs):
        print(f"fp16 enc[{i}]: max={feat.abs().max().item():.1f}  nan={torch.isnan(feat).any().item()}")

    # 3. full fp16
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        pred_fp16 = model(raw)
    print("fp16 full  :", pred_fp16.min().item(), pred_fp16.max().item(), "nan:", torch.isnan(pred_fp16).any().item())

    break

In [ ]:
import torch
print(torch.__version__)

In [ ]:
import micro_sam
print(micro_sam.__version__)

In [ ]:
import torch

# grab one sample from your loader
raw, _ = next(iter(train_loader))
raw = raw[:1].to(device)   # (1, 3, H, W), values in [0, 255]

model.eval()
execution_log = []

def _make_hook(name):
    def hook(module, inp, out):
        t = out if isinstance(out, torch.Tensor) else \
            next((x for x in out if isinstance(x, torch.Tensor)), None) \
            if isinstance(out, (list, tuple)) else None
        if t is None:
            return
        has_nan = torch.isnan(t).any().item()
        execution_log.append({
            "name": name,
            "nan":  has_nan,
            "max":  float(t.abs().max()) if not has_nan else float("nan"),
            "shape": tuple(t.shape),
        })
    return hook

hooks = [mod.register_forward_hook(_make_hook(name))
         for name, mod in model.named_modules() if name]

with torch.no_grad():
    with torch.autocast("cuda", dtype=torch.float16):
        _ = model(raw)

for h in hooks:
    h.remove()

idx = next((i for i, e in enumerate(execution_log) if e["nan"]), None)

if idx is None:
    print("No NaN detected in fp16 forward pass")
else:
    print(f"First NaN at: {execution_log[idx]['name']}\n")
    for e in execution_log[max(0, idx - 4) : idx + 6]:
        flag = " <<< FIRST NaN" if e is execution_log[idx] else ""
        print(f"  {'NaN' if e['nan'] else 'ok ':3s}  "
              f"max={e['max']:>10.1f}  "
              f"{str(e['shape']):35s}  "
              f"{e['name']}{flag}")

In [ ]:
target = dict(model.named_modules())["deconv1.block.1.block"]
print(type(target))
print(target)

log2 = []
def io_hook(module, inp, out):
    t_in = inp[0] if isinstance(inp, tuple) else inp
    log2.append({
        "in_nan":  torch.isnan(t_in).any().item(),
        "in_max":  float(t_in.abs().max()) if not torch.isnan(t_in).any() else float("nan"),
        "out_nan": torch.isnan(out).any().item(),
        "out_max": float(out.abs().max()) if not torch.isnan(out).any() else float("nan"),
    })

h = target.register_forward_hook(io_hook)
with torch.no_grad():
    with torch.autocast("cuda", dtype=torch.float16):
        _ = model(raw.to(device))
h.remove()

e = log2[0]
print(f"input:  nan={e['in_nan']}  max={e['in_max']:.4f}")
print(f"output: nan={e['out_nan']}  max={e['out_max']}")

It happens in the decoder's convolutional block.

### Adjust CUDNN algorithm

In [ ]:
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = True

# with torch.no_grad():
#     with torch.autocast("cuda", dtype=torch.float16):
#         out = model(raw.to(device))

# print(f"deterministic fp16: nan={torch.isnan(out).any().item()}")

# restore
# torch.backends.cudnn.deterministic = False

Did not help. All resulted in NaNs.

### Try BF16 for computation

Worked.

In [ ]:
import torch

model.eval()
raw_sample, _ = next(iter(train_loader))
raw_sample = raw_sample[:1].to(device)

conv = dict(model.named_modules())["deconv1.block.1.block"]
hook_log = {}

def make_hook(label):
    def hook(module, inp, out):
        hook_log[label] = {
            "in_max":  float(inp[0].abs().max()),
            "out_nan": torch.isnan(out).any().item(),
            "out_max": float(out.abs().max()) if not torch.isnan(out).any() else float("nan"),
        }
    return hook

for dtype, label in [(torch.float16, "fp16"), (torch.bfloat16, "bf16")]:
    h = conv.register_forward_hook(make_hook(label))
    with torch.no_grad():
        with torch.autocast("cuda", dtype=dtype):
            full_out = model(raw_sample)
    h.remove()
    e = hook_log[label]
    print(f"{label}  deconv1 Conv2d(512,512,3Ã3):  "
          f"in_max={e['in_max']:.4f}  out_nan={e['out_nan']}  out_max={e['out_max']:.4f}")
    print(f"      full model output:  nan={torch.isnan(full_out).any().item()}")

In [ ]:
import torch
from micro_sam.training.training import _get_mixed_precision_dtype

# --- 1. dtype detection ---
device = torch.device("cuda")
dtype = _get_mixed_precision_dtype(device)
print(f"GPU:   {torch.cuda.get_device_name(0)}")
print(f"dtype: {dtype}  (expected torch.bfloat16 for GTX 1660 Ti)")

# --- 2. compare fp16 vs detected dtype on the failing layer ---
conv = dict(model.named_modules())["deconv1.block.1.block"]
model.eval()
raw_sample, _ = next(iter(train_loader))
raw_sample = raw_sample[:1].to(device)

results = {}
for test_dtype, label in [(torch.float16, "fp16"), (dtype, f"auto ({dtype})")]:
    log = {}
    def make_hook(lbl):
        def hook(m, inp, out):
            log[lbl] = {
                "in_max":  float(inp[0].abs().max()),
                "out_nan": torch.isnan(out).any().item(),
            }
        return hook
    h = conv.register_forward_hook(make_hook(label))
    with torch.no_grad():
        with torch.autocast("cuda", dtype=test_dtype):
            full_out = model(raw_sample)
    h.remove()
    e = log[label]
    print(f"\n{label}")
    print(f"  deconv1 Conv2d(512,512,3x3): in_max={e['in_max']:.4f}  out_nan={e['out_nan']}")
    print(f"  full model output:           nan={torch.isnan(full_out).any().item()}")